# AI_TRANSLATE — Multilingual Product Catalog

`AI_TRANSLATE` translates text between languages in a single SQL call.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.AI_TRANSLATE` |
| **Data** | 16 product descriptions in 12 languages |
| **Use-case** | Centralize multilingual content into a single language for analysis |


## Step 1 — Environment & Table Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE TABLE product_catalog_multilingual (
    entry_id    INT AUTOINCREMENT,
    sku         VARCHAR(20),
    category    VARCHAR(100),
    language    VARCHAR(10),
    description TEXT
);

In [ ]:
INSERT INTO product_catalog_multilingual (sku, category, language, description) VALUES
('SKU-001', 'Electronics', 'en', 'High-performance wireless headphones with active noise cancellation and 30-hour battery life. Features premium drivers for studio-quality sound.'),
('SKU-001', 'Electronics', 'es', 'Auriculares inalambricos de alto rendimiento con cancelacion activa de ruido y 30 horas de bateria. Con drivers premium para sonido de calidad de estudio.'),
('SKU-001', 'Electronics', 'fr', 'Casque sans fil haute performance avec reduction active du bruit et 30 heures de batterie. Equiped de transducteurs premium pour un son qualite studio.'),
('SKU-001', 'Electronics', 'de', 'Hochleistungs-Funkkopfhorer mit aktiver Gerauschunterdruckung und 30 Stunden Akkulaufzeit. Mit Premium-Treibern fur Studioklang-Qualitat.'),
('SKU-002', 'Kitchen', 'en', 'Smart coffee maker with built-in grinder, programmable brewing schedules, and WiFi connectivity. Makes up to 12 cups with precision temperature control.'),
('SKU-002', 'Kitchen', 'ja', 'ビルトイングラインダー、プログラム可能な抽出スケジュール、WiFi接続を備えたスマートコーヒーメーカー。精密温度制御で最大12杯まで抽出可能。'),
('SKU-002', 'Kitchen', 'zh', '内置研磨机、可编程冲泡计划和WiFi连接的智能咖啡机。精确温控，最多可冲泡12杯。'),
('SKU-002', 'Kitchen', 'it', 'Macchina da caffe intelligente con macinino integrato, programmi di erogazione programmabili e connettivita WiFi. Fino a 12 tazze con controllo preciso della temperatura.'),
('SKU-003', 'Sports', 'en', 'Ultra-lightweight running shoes with responsive foam cushioning and breathable mesh upper. Designed for long-distance comfort and speed.'),
('SKU-003', 'Sports', 'pt', 'Tenis de corrida ultraleve com amortecimento de espuma responsiva e cabedal em malha respiravel. Projetado para conforto e velocidade em longas distancias.'),
('SKU-003', 'Sports', 'nl', 'Ultralichte hardloopschoenen met responsieve schuimdemping en ademend mesh bovenwerk. Ontworpen voor comfort en snelheid op lange afstanden.'),
('SKU-003', 'Sports', 'ko', '반응성 폼 쿠셔닝과 통기성 메쉬 어퍼를 갖춘 초경량 런닝화. 장거리 편안함과 속도를 위해 설계되었습니다.'),
('SKU-004', 'Furniture', 'en', 'Ergonomic standing desk with electric height adjustment, memory presets, and cable management system. Supports up to 150kg with whisper-quiet motor.'),
('SKU-004', 'Furniture', 'pl', 'Ergonomiczne biurko do pracy stojaco-siedzacej z elektryczna regulacja wysokosci, zapamietywanymi pozycjami i systemem prowadzenia kabli. Udzwig do 150 kg z ultra-cichym silnikiem.'),
('SKU-004', 'Furniture', 'ar', 'مكتب مريح بتعديل ارتفاع كهربائي مع اعدادات محفوظة ونظام ادارة الكابلات. يتحمل حتى 150 كجم مع محرك هادئ للغاية.'),
('SKU-004', 'Furniture', 'de', 'Ergonomischer Steh-Schreibtisch mit elektrischer Hohenverstellung, Speichervoreinstellungen und Kabelmanagement. Tragt bis zu 150 kg mit flusterleisem Motor.');

In [ ]:
SELECT * FROM product_catalog_multilingual LIMIT 5;

## Step 2 — Translate to English

Normalize all descriptions to English for unified analysis.

In [ ]:
SELECT
    sku,
    category,
    language AS source_lang,
    SNOWFLAKE.CORTEX.AI_TRANSLATE(description, language, 'en') AS english_description
FROM product_catalog_multilingual
WHERE language != 'en'
ORDER BY sku;

## Step 3 — Cross-Language Quality Check

In [ ]:
WITH originals AS (
    SELECT sku, description AS en_description
    FROM product_catalog_multilingual WHERE language = 'en'
),
translations AS (
    SELECT
        sku, language,
        SNOWFLAKE.CORTEX.AI_TRANSLATE(description, language, 'en') AS back_translated
    FROM product_catalog_multilingual WHERE language != 'en'
)
SELECT
    t.sku, t.language,
    LEFT(o.en_description, 60)  AS original_en,
    LEFT(t.back_translated, 60) AS back_translated_en
FROM translations t
JOIN originals o ON t.sku = o.sku
ORDER BY t.sku, t.language;

## Key Takeaways

- `AI_TRANSLATE` handles 10+ languages with a single function call.
- Use back-translation to quality-check existing multilingual content.
- Combine with `AI_SENTIMENT` to analyze feedback across all languages.
- Great for e-commerce catalogs, support tickets, and documentation.
